# What the learned encoder consumes and generates — a step-by-step walk-through

**Goal of this notebook (Phase-6 Workstream A).** Take one example `.pdb`, process it stage-by-stage into the
exact artefact the learned encoder consumes, run inference, and show the **shape of what it produces**.

**The one-sentence answer (proved below):** the encoder maps a protein chain's *surface* to a
**per-surface-atom 32-D embedding vector** — an *embedding field over the molecular surface*, **not** a
per-atom binding score. A binding "score" is a separate, *downstream, pairwise* computation between two chains'
embeddings via a learned bilinear form `T`.

Pipeline: **PDB → molecular surface (MSMS) + chemistry → heterogeneous graph → encoder → per-atom embedding →
(pairwise) binding score.**

In [ ]:
import os, numpy as np, torch
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa
from plyfile import PlyData
%matplotlib inline

# --- masif-graph pipeline pieces (the same code used in Phase 5) ---
from masif_graph.io.reference import load_complex, PDB_DIR, parse_heavy_atoms
from masif_graph.surface.atoms import build_surface_atoms
from masif_graph.graph.hetero import build_hetero_graph
from masif_graph.p4.dataset import hetero_to_dict, load_chain_graph, D_AA, D_VV, D_VA
from masif_graph.p4.encoder import HeteroEncoder
from masif_graph.p4.objective import Complementarity, normalize

torch.set_num_threads(8)
REF   = "/scratch/ymeng/masif-graph/masif-neosurf-af2/masif/data/masif_ppi_search"
NPZ   = "/work/upthomae/Meng/phase5/npz"
CKPT  = "/work/upthomae/Meng/phase4/ret_full_ctr_best.pt"
EX    = "1G60_A_B"           # example complex: chain A (query) + chain B (its true partner)
BLUE, ORANGE = "#0072B2", "#E69F00"

# --- load the trained encoder + the bilinear complementarity form T ---
ck  = torch.load(CKPT, map_location="cpu"); cfg = ck.get("cfg", {})
def make_encoder(f_atom, f_vert):
    e = HeteroEncoder(f_atom, f_vert, D_AA, D_VV, D_VA,
                      d=cfg.get("d",64), d_out=cfg.get("d_out",32), n_layers=cfg.get("layers",4))
    e.load_state_dict(ck["enc"]); e.eval(); return e
comp = Complementarity(cfg.get("d_out",32), tau_init=cfg.get("tau",0.1)); comp.load_state_dict(ck["comp"])
T = comp.T.detach()
print("checkpoint:", os.path.basename(CKPT), "| d_out =", cfg.get("d_out",32), "| bilinear T:", tuple(T.shape))
ENC = make_encoder(14, 4)   # one shared encoder instance (atom feats=14, vertex channels=4)
print("shared encoder ready")

## Stage 0 — the input: a `.pdb` structure
The raw artefact is an experimental (or AI-predicted) protein structure. Our example is PDB **1G60**, a
two-chain complex; we treat chain **A** as the query and chain **B** as its true binding partner.

In [ ]:
pdb_id = EX.split("_")[0]
raw_pdb = f"{REF}/data_preparation/00-raw_pdbs/{pdb_id}.pdb"
print("example PDB file:", raw_pdb)

# heavy atoms of each chain (via the same parser the pipeline uses on the extracted per-chain PDBs)
p1, p2 = load_complex(EX)   # p1 = chain A, p2 = chain B (surfaces + descriptors already precomputed)
print(f"chain A: {p1.n_atom} heavy atoms, {p1.n_vert} surface vertices")
print(f"chain B: {p2.n_atom} heavy atoms, {p2.n_vert} surface vertices")

fig = plt.figure(figsize=(6,5)); ax = fig.add_subplot(111, projection="3d")
ax.scatter(*p1.atom_coords.T, s=3, c=BLUE,   alpha=.5, label="chain A (query)")
ax.scatter(*p2.atom_coords.T, s=3, c=ORANGE, alpha=.5, label="chain B (partner)")
ax.set_title(f"{EX}: heavy atoms of the two chains"); ax.legend(); ax.set_axis_off(); plt.show()

## Stage 1 — molecular surface + MaSIF chemistry channels
The reference `.sif` pipeline (MSMS + APBS) triangulates the solvent-excluded **molecular surface** and
computes four per-vertex chemical/geometric channels: **shape index, H-bond potential, Poisson–Boltzmann
charge, hydrophobicity**. This is MaSIF's surface representation.

*(Produced offline by `01-pdb_extract_and_triangulate.py` inside the `.sif`; here we load and visualize it.)*

In [ ]:
ply = PlyData.read(f"{REF}/data_preparation/01-benchmark_surfaces/{pdb_id}_A.ply")
V   = ply["vertex"].data
vxyz = np.column_stack([V["x"], V["y"], V["z"]])
channels = {"H-bond": V["hbond"], "charge": V["charge"], "hydrophobicity": V["hphob"]}
# shape-index comes from the precompute; read it from the hetero-graph vert_feat below. For now show 3 channels.
fig, axes = plt.subplots(1, 3, figsize=(15,4.5), subplot_kw={"projection":"3d"})
for ax,(name,val) in zip(axes, channels.items()):
    p = ax.scatter(*vxyz.T, c=val, s=2, cmap="coolwarm")
    ax.set_title(f"surface colored by {name}"); ax.set_axis_off(); fig.colorbar(p, ax=ax, shrink=.5)
fig.suptitle(f"{EX} chain A — molecular surface ({len(vxyz)} vertices) with MaSIF channels", weight="bold")
plt.tight_layout(); plt.show()

## Stage 2 — the heterogeneous graph (the encoder's actual input format)
The encoder does **not** read the surface directly. Each chain is turned into a **HeteroSurfaceGraph** with two
node types and three SE(3)-invariant edge types:
- **surface heavy-atom nodes** — one per solvent-exposed heavy atom (14-D invariant chemistry features);
- **surface-vertex nodes** — the mesh vertices (4-D = [shape-index, H-bond, charge, hydrophobicity]);
- **edges:** atom–atom covalent bonds, vertex–vertex mesh adjacency, vertex–atom proximity.

We build it **live** here (same functions as Phase-5 preprocessing) to show the exact structure.

In [ ]:
surf1 = build_surface_atoms(p1.verts, p1.atom_coords, p1.atom_element, p1.atom_resid,
                            p1.desc_straight, p1.desc_flipped, ops=("mean",))
g1 = build_hetero_graph(p1, surf1, f"{PDB_DIR}/{p1.pdb_id}_{p1.chain_ids}.pdb")
print("HeteroSurfaceGraph for chain A:")
print(f"  atom nodes : {g1.atom_feat.shape}   (surface heavy atoms among them: {len(surf1.coord)})")
print(f"  vertex nodes: {g1.vert_feat.shape}   [shape-index, hbond, charge, hphob]")
print(f"  atom-atom covalent edges: {g1.aa_edge.shape[1]}")
print(f"  vertex-vertex mesh edges: {g1.vv_edge.shape[1]}")
print(f"  vertex-atom edges       : {len(g1.va_v)}")

# visualize: surface heavy atoms + their covalent bonds (the chemistry graph)
fig = plt.figure(figsize=(6.5,5.5)); ax = fig.add_subplot(111, projection="3d")
ac = g1.atom_feat  # nodes; coords are p1.atom_coords
co = p1.atom_coords
for s,d in g1.aa_edge.T[:4000]:
    ax.plot(*co[[s,d]].T, c="#B0B4B8", lw=.4, alpha=.5)
ax.scatter(*surf1.coord.T, s=6, c=BLUE, label=f"{len(surf1.coord)} surface heavy atoms")
ax.set_title("chain A: surface heavy-atom nodes + covalent (chem) edges"); ax.legend(); ax.set_axis_off(); plt.show()

# the encoder-input dict (tensors) — this is literally what enc() receives
gd1 = hetero_to_dict(g1)
print("\nencoder input dict keys:", list(gd1.keys()))

## Stage 3 — run the encoder → the output artefact
`encoder(graph) → z`. **The output is one 32-D vector per surface heavy atom**, L2-normalized. It is an
*embedding field over the surface*, **not** a scalar and **not** a per-atom binding score.

In [ ]:
with torch.no_grad():
    z1 = normalize(ENC(gd1))          # (n_surf_atoms, 32)          # (n_surf_atoms, 32)
print(f">>> ENCODER OUTPUT: z has shape {tuple(z1.shape)}")
print(f"    = {z1.shape[0]} surface heavy atoms x {z1.shape[1]}-D embedding; each row L2-normalized "
      f"(||row0|| = {z1[0].norm():.3f}).")
print("    This is the artefact. It is NOT a per-atom score.")

# visualize the embedding field: PCA of the 32-D vectors -> RGB, painted on the surface atoms
from numpy.linalg import svd
Z = z1.numpy(); Zc = Z - Z.mean(0)
U,S,Vt = svd(Zc, full_matrices=False); rgb = Zc @ Vt[:3].T
rgb = (rgb - rgb.min(0)) / (np.ptp(rgb, 0) + 1e-9)
fig = plt.figure(figsize=(6.5,5.5)); ax = fig.add_subplot(111, projection="3d")
ax.scatter(*surf1.coord.T, c=rgb, s=10)
ax.set_title("chain A surface — learned 32-D embedding (PCA\u2192RGB per surface atom)"); ax.set_axis_off(); plt.show()

## Stage 4 — from embeddings to a binding score (downstream, pairwise)
The binding signal is computed **between two chains**, not inside one. For a query atom `i` (chain A) and a DB
atom `j` (chain B), the learned complementarity is the bilinear `zᵢᵀ T zⱼ`. A chain-vs-chain retrieval score
aggregates it over the interface patch: `score = medianᵢ maxⱼ zᵢᵀ T zⱼ`. A true partner scores higher than a
decoy — that is what drives retrieval.

In [ ]:
def embed_npz(cid, pid):
    with torch.no_grad():
        return normalize(ENC(load_chain_graph(f"{NPZ}/{cid}__holo__{pid}.npz")))
def patch(cid, pid):
    z = embed_npz(cid, pid)
    con = np.load(f"{NPZ}/{cid}__contacts.npz")["pos"]
    return z[np.unique(con[:, 0 if pid=="p1" else 1])]
def scr(zq, zd):                              # median over query of max over db of z_i^T T z_j
    with torch.no_grad(): return float((zq @ T @ zd.t()).max(1).values.median())

qA = patch(EX, "p1")                          # query = 1G60 chain A interface patch
zB = patch(EX, "p2")                          # true partner = chain B

# (a) the raw pairwise complementarity matrix vs the TRUE partner
M = (qA @ T @ zB.t()).numpy()
fig, ax = plt.subplots(figsize=(6,4.2))
im = ax.imshow(M, aspect="auto", cmap="coolwarm"); fig.colorbar(im, shrink=.7)
ax.set_xlabel("chain B interface atoms (j)"); ax.set_ylabel("chain A interface atoms (i)")
ax.set_title("pairwise complementarity  $z_i^T T z_j$  (query A vs TRUE partner B)"); plt.tight_layout(); plt.show()

# (b) retrieval: rank the true partner in a real database of candidate chains (held-out eval set)
ids = [l.strip() for l in open("/scratch/ymeng/masif-graph/logs/phase5/eval_sc304_clean_vs_enc.txt") if l.strip()]
db = [(f"{EX[:4]}_B  (TRUE partner)", zB)]
for cid in ids[:45]:
    if cid == EX: continue
    for pid,l in (("p1","A"),("p2","B")):
        try: db.append((f"{cid[:4]}_{l}", patch(cid, pid)))
        except Exception: pass
sc = sorted([(l, scr(qA, zd)) for l,zd in db], key=lambda x:-x[1])
rank = 1 + [i for i,(l,_) in enumerate(sc) if "TRUE" in l][0]
print(f"query = {EX[:4]} chain A;  database = {len(db)} candidate partner chains (held-out)")
print(f">>> TRUE partner (chain B) retrieval rank: {rank} / {len(db)}")
top = sc[:15]                                  # show the top-15 for readability
labels=[l for l,_ in top]; vals=[v for _,v in top]
colors=[BLUE if "TRUE" in l else "#B0B4B8" for l in labels]
fig, ax = plt.subplots(figsize=(7,4.6))
ax.barh(np.arange(len(vals))[::-1], vals, color=colors)
ax.set_yticks(np.arange(len(vals))[::-1]); ax.set_yticklabels(labels, fontsize=8)
ax.set_xlabel("retrieval score  (median-of-max  $z^T T z$)"); ax.set_xlim(min(vals)-.03, max(vals)+.01)
ax.set_title(f"Retrieval over {len(db)} candidates: TRUE partner ranks #{rank}  (top 15 shown)")
plt.tight_layout(); plt.show()

## Summary — the I/O contract in one sentence

**Consumes** a per-chain *HeteroSurfaceGraph* (surface heavy-atom nodes + surface-vertex nodes + 3 invariant
edge types). **Emits** a **per-surface-atom 32-D embedding field** (`(n_surf_atoms, 32)`, L2-normalized) — not
a per-atom score. A **binding score is downstream and pairwise**: `medianᵢ maxⱼ zᵢᵀ T zⱼ` between two chains'
embeddings, which is what ranks a database of candidate partners.

This is the contract that Phase-6 Workstream C must extend to **ligands** (add ligand atoms as graph nodes so
the embedding field also covers a bound small molecule — the neosurface).